# LAB | Abstractive Question Answering

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [1]:
!pip install --upgrade pip setuptools wheel
!pip install numpy pandas pyarrow datasets transformers sentence-transformers huggingface_hub pinecone-client tqdm -U

# Load and Prepare Dataset

Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [2]:
# load the dataset from huggingface in streaming mode and shuffle it
from datasets import load_dataset

wiki_data = load_dataset(
    'vblagoje/lfqa_support_docs',
    split='train',
    streaming=True,
    trust_remote_code=True
).shuffle(seed=960)

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'vblagoje/lfqa_support_docs' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Repo card metadata block was not found. Setting CardData to empty.


We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [3]:
# show the contents of a single document in the dataset
next(iter(wiki_data))


{'id': '6pt2tc',
 'input': 'why is the word "reich" always used in texts about germany, instead of translating it?',
 'output': [{'answer': 'Because the word "realm" (the closest translation) is too general. Even German has multiple words for "realm of a king" vs "realm of an emperor" vs "realm of someone else".\n\nBy saying "the Nazi realm", what do we mean? Do we mean Germany and Austria, where the Nazis actually ruled directly? Do we mean those two AND all the surrounding countries they conquered, where they held control though puppets?\n\nWhereas if we say "the Reich", we know immediately what was meant. They themselves distinguished between "Reich" and "occupied territories".',
   'provenance': None},
  {'answer': None,
   'provenance': [{'wikipedia_id': '25519',
     'title': 'Reich',
     'section': '',
     'start_paragraph_id': 1,
     'start_character': 0,
     'end_paragraph_id': 1,
     'end_character': 399,
     'text': 'Reich (; ) is a German word analogous in meaning to 

In [4]:
# The 'wiki_snippets' dataset does not have 'section_title', so we will proceed without this specific filter
history = []
limit = 50000
counter = 0

for d in wiki_data:
    if counter >= limit:
        break
# Each row 'd' has a list of answers ('output')
    for out in d.get('output', []):
        # Each answer has a list of snippets ('provenance')
        for s in out.get('provenance', []) or []:
            if s.get('section') and "History" in s['section']:
                # We save the data with the keys your NEXT cell expects
                history.append({
                    'article': s['title'],
                    'section': s['section'],
                    'passage': s['text']
                })
                counter += 1
            if counter >= limit: break
        if counter >= limit: break
print(f"Step 1 Complete: 'history' list now has {len(history)} items.")

Step 1 Complete: 'history' list now has 50000 items.


Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [5]:
from tqdm.auto import tqdm  # progress bar

total_doc_count = 50000 

docs = []
# 1. Iteramos sobre la lista 'history' que creamos en la celda anterior
for d in tqdm(history, total=total_doc_count):
    # 2. Extraemos los campos y los guardamos en un diccionario
    # IMPORTANTE: Eliminamos las comas al final de las líneas para evitar que se creen tuplas
    docs.append({
        'title': d['article'],   # Lo guardamos como 'title' para la metadata de Pinecone
        'section': d['section'],
        'passage': d['passage']  # Lo guardamos como 'passage' para que el Retriever lo lea
    })

# 3. Verificamos cuántos documentos hemos guardado realmente
print(f"Éxito: Se han recolectado {len(docs)} pasajes históricos.")

  0%|          | 0/50000 [00:00<?, ?it/s]

Éxito: Se han recolectado 50000 pasajes históricos.


In [6]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)
df.head()

,title,section,passage
0,Active redundancy,Section::::History.\n,Satellites placed into orbit around the earth ...
1,Wisconsin River,Section::::History.\n,As American soldiers and officials traveled th...
2,Quapaw,Section::::History.\n,English speakers tried to adapt French names t...
3,Europe (magazine),Section::::History.\n,"Since the 1950s, ""Europe"" has issued thematic ..."
4,"Hof, Bavaria",Section::::History.:Modern Era.\n,"In 1989 thousands of East German citizens, who..."


# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [ ]:
# !pip uninstall -y pinecone-client pinecone -q
# !pip install "pinecone[grpc]" -U -q  # Latest v5 with gRPC

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-pinecone 0.2.13 requires pinecone[asyncio]<8.0.0,>=6.0.0, but you have pinecone 8.1.0 which is incompatible.


In [8]:
# from google.colab import userdata
import os
from pinecone import Pinecone
from pinecone import ServerlessSpec

# initialize connection to pinecone (get API key at app.pinecone.io)
pinecone_api_key = os.getenv("PINECONE_API_KEY")



Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [10]:
spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
) 
# connect to pinecone environment
pc = Pinecone(api_key=pinecone_api_key, environment=spec.region)

PineconeConfigurationError: You haven't specified an API key. Please either set the PINECONE_API_KEY environment variable or pass the 'api_key' keyword argument to the Pinecone client constructor.

Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [ ]:
index_name = 'abstractive-question-answerin' #give your index a meaningful name

import time
# Check if index exists & create with ServerlessSpec
if index_name not in pc.list_indexes().names():
    #create the index if it does not exist
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=spec)

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# load the retriever model from huggingface model hub
retriever = SentenceTransformer("flax-sentence-embeddings/all_datasets_v3_mpnet-base", device=device)
retriever

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/591 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [ ]:
index = pc.Index(index_name)
# we will use batches of 64
batch_size = 64

#You will create embedding for the passage_text variable and be use to include the meta data in each batch
for i in tqdm(range(0, len(df), batch_size)):
    # find end of batch
    END = min(i + batch_size, len(df))
    # extract batch
    batch = df.iloc[i:END]
    # generate embeddings for batch
    embeddings = retriever.encode(batch['passage'].tolist(), show_progress_bar=False)
    
    current_batch_vectors = []

    # create list of vectors with metadata

    for j, (idx, row) in enumerate(batch.iterrows()):
        current_batch_vectors.append({
            "id": f"doc_{row.idx}",
            "values": embeddings[j].tolist(),
            "metadata": {
                "title": row.title,
                "passage": row.passage
            }
        })

    index.upsert(vectors=current_batch_vectors)
print("\n--- Estadísticas Finales ---")
print(index.describe_index_stats())

0it [00:00, ?it/s]


--- Estadísticas Finales ---
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '150',
                                    'content-type': 'application/json',
                                    'date': 'Wed, 25 Feb 2026 17:06:51 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '6',
                                    'x-pinecone-request-latency-ms': '6',
                                    'x-pinecone-response-duration-ms': '8'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

# load bart tokenizer and model from huggingface
tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [ ]:
def query_pinecone(query, top_k):
    # generate embeddings for the query
    xq = retriever.encode(query, show_progress_bar=False)
    # search pinecone index for context passage with the answer
    xc = index.query(
        vector=xq.tolist(),
        top_k=top_k,
        include_metadata=True
    )
    return xc

In [ ]:
def format_query(query, context):
    # extract passage_text from Pinecone search result and add the <P> tag
    context = [f"<P> {m['metadata']['passage']}" for m in context]
    # concatinate all context passages
    context = " ".join(context) 
    # contcatinate the query and context passages
    query = f"{query} {context}"
    return query

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [ ]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

QueryResponse(matches=[], namespace='', usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Wed, 25 Feb 2026 17:04:45 GMT', 'content-type': 'application/json', 'content-length': '66', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '110', 'x-envoy-upstream-service-time': '40', 'x-pinecone-response-duration-ms': '112', 'grpc-status': '0', 'server': 'envoy'}})

In [ ]:
from pprint import pprint

In [ ]:
# format the query in the form generator expects the input
query = format_query(query, result['matches'])
pprint(query)

'when was the first electric power system built? '


The output looks great. Now let's write a function to generate answers.

In [ ]:
def generate_answer(query):
    # tokenize the query to get input_ids
    inputs = tokenizer([query], max_length=1024, return_tensors="pt").to(device)
    # use generator to predict output ids
    ids = generator.generate(inputs["input_ids"], num_beams=2, min_length=20, max_length=40)
    # use tokenizer to decode the output ids
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return pprint(answer)

In [ ]:
generate_answer(query)

As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [ ]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

To confirm that this answer is correct, we can check the contexts used to generate the answer.

In [ ]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [ ]:
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

In [ ]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

Let’s finish with a final few questions.

In [ ]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

In [ ]:
query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

In [ ]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

As we can see, the model can generate some decent answers.

#### Add a few more questions